In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Ensure the new directory exists
os.makedirs('../dataset/processed_merged', exist_ok=True)

# Load the new generalized datasets
df_long = pd.read_csv('../dataset/raw/oasis_longitudinal_2.csv')
df_cross = pd.read_csv('../dataset/raw/oasis_cross-sectional_2.csv')

print(f"Original Longitudinal Shape: {df_long.shape}")
print(f"Original Cross-Sectional Shape: {df_cross.shape}")

Original Longitudinal Shape: (373, 15)
Original Cross-Sectional Shape: (436, 12)


In [2]:
# --- 1. Prepare Longitudinal Data ---
df_long['Group'] = df_long['Group'].replace(['Converted'], ['Demented'])
df_long['Group'] = df_long['Group'].map({'Nondemented': 0, 'Demented': 1})
# Drop useless/leaky columns
df_long = df_long.drop(['Subject ID', 'MRI ID', 'Hand', 'Visit', 'MR Delay'], axis=1)

# --- 2. Prepare Cross-Sectional Data ---
# Drop young patients (keep Age >= 60 to match longitudinal demographics)
df_cross = df_cross[df_cross['Age'] >= 60].copy()

# Create the missing 'Group' target using CDR (CDR > 0 means Dementia)
df_cross['Group'] = np.where(df_cross['CDR'] > 0, 1, 0)

# Align column names and drop useless columns
df_cross = df_cross.rename(columns={'Educ': 'EDUC'}) # Match capitalization
df_cross = df_cross.drop(['ID', 'Hand', 'Delay'], axis=1)

# --- 3. Merge and De-duplicate ---
# Stack them on top of each other
merged_df = pd.concat([df_long, df_cross], axis=0, ignore_index=True)

# Drop duplicates (patients who likely participated in both studies based on exact biometrics)
merged_df = merged_df.drop_duplicates(subset=['Age', 'EDUC', 'eTIV', 'nWBV', 'M/F'])

# Map Gender to Binary
merged_df['M/F'] = merged_df['M/F'].map({'M': 1, 'F': 0})

print(f"Final Merged Dataset Shape: {merged_df.shape}")
print(merged_df['Group'].value_counts())

Final Merged Dataset Shape: (571, 10)
Group
0    288
1    283
Name: count, dtype: int64


In [3]:
# 1. Split
X = merged_df.drop(['Group', 'CDR'], axis=1)
y = merged_df['Group']
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Impute (Median)
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 3. Scale (StandardScaler)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X_train_imputed.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X_test_imputed.columns)

# 4. SMOTE (Class Balancing on Training Data)
smote = SMOTE(random_state=42)
X_train_resampled, Y_train_resampled = smote.fit_resample(X_train_scaled, Y_train)

# 5. Export to the NEW folder
X_train_resampled.to_csv('../dataset/processed_merged/X_train.csv', index=False)
X_test_scaled.to_csv('../dataset/processed_merged/X_test.csv', index=False)
Y_train_resampled.to_csv('../dataset/processed_merged/Y_train.csv', index=False)
Y_test.to_csv('../dataset/processed_merged/Y_test.csv', index=False)

print("Merged Data Preprocessing Complete! Saved to 'processed_merged' directory.")

Merged Data Preprocessing Complete! Saved to 'processed_merged' directory.


In [4]:
print(X_train.shape)
print(Y_train_resampled.value_counts())

(456, 8)
Group
1    230
0    230
Name: count, dtype: int64
